# BrainScan V2 — Complete YOLOv8 Detection Notebook

Local-first workflow: dataset inspection → annotation audit → visualization → train/valid/test preparation → `data.yaml` → YOLOv8 training → validation/test evaluation → MLflow tracking → model export.

**Dataset expected:** original folder containing `Train/` and `Val/`; inside each class folder are `images/` and `labels/`. Set `DATASET_ROOT` below to the directory that directly contains `Train` and `Val`.

The split follows the earlier project: 85% of matched original Train images for training, 15% for test, and matched original Val images for validation. This is image-level splitting, not patient-level splitting.

## 1. Install dependencies
Run once. Restart the kernel if necessary.

In [ ]:
%pip install -U ultralytics mlflow scikit-learn pandas matplotlib pillow pyyaml

## 2. Imports and configuration
Adjust `DATASET_ROOT` if your extracted source dataset is elsewhere. Use `DEVICE='cpu'` if you do not have a CUDA-compatible GPU.

In [ ]:
from pathlib import Path
import random, shutil, json, re
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image, ImageDraw
from sklearn.model_selection import train_test_split
from ultralytics import YOLO
import yaml
import mlflow

SEED = 42
IMG_SIZE = 640
EPOCHS = 50
BATCH_SIZE = 8
WORKERS = 2
DEVICE = "cpu"   # Change to 0 only if your local machine has a compatible CUDA GPU.

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "models").exists() and (PROJECT_ROOT.parent / "models").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

# This folder must directly contain Train/ and Val/.
DATASET_ROOT = PROJECT_ROOT / "kaggle_upload"
OUTPUT_DATASET = PROJECT_ROOT / "data" / "brain_tumor_yolo"
RUNS_DIR = PROJECT_ROOT / "mlartifacts" / "yolo_runs"
EXPORT_DIR = PROJECT_ROOT / "models" / "final" / "yolo_export"

CLASS_NAMES = ["Glioma", "Meningioma", "Pituitary"]
# Mapping used in the earlier notebook. VERIFY with the dataset's official class definition.
CLASS_MAPPING = {0: 0, 1: 1, 3: 2}

random.seed(SEED)
np.random.seed(SEED)

print("Project:", PROJECT_ROOT)
print("Dataset search folder:", DATASET_ROOT)

## 3. Locate the original dataset

In [ ]:
def find_dataset_root(start):
    if (start / "Train").is_dir() and (start / "Val").is_dir():
        return start
    if not start.exists():
        raise FileNotFoundError(f"Folder not found: {start}. Extract the dataset here or update DATASET_ROOT.")
    for train_dir in start.rglob("Train"):
        candidate = train_dir.parent
        if (candidate / "Val").is_dir():
            return candidate
    raise FileNotFoundError(f"Could not find Train/ and Val/ under {start}")

DATASET_ROOT = find_dataset_root(DATASET_ROOT)
print("Dataset root:", DATASET_ROOT)

## 4. Inspect source image and annotation counts

In [ ]:
IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp"}

def list_images(folder):
    return sorted(p for p in folder.rglob("*") if p.is_file() and p.suffix.lower() in IMAGE_EXTENSIONS)

for split in ["Train", "Val"]:
    print(f"\\n{split}")
    for class_dir in sorted(p for p in (DATASET_ROOT / split).iterdir() if p.is_dir()):
        images_dir, labels_dir = class_dir / "images", class_dir / "labels"
        images = list_images(images_dir) if images_dir.exists() else []
        labels = list(labels_dir.glob("*.txt")) if labels_dir.exists() else []
        print(f"{class_dir.name:15} images={len(images):4} labels={len(labels):4}")

## 5. Audit image-label matching and YOLO annotation validity
A YOLO annotation row must have five values: `class_id x_center y_center width height`. Coordinates must be normalized to [0, 1]. Empty files are allowed as negative images.

In [ ]:
def audit_split(split_name):
    records, missing, orphan, invalid = [], [], [], []
    class_counts = Counter()
    empty_count = 0

    for class_dir in sorted(p for p in (DATASET_ROOT / split_name).iterdir() if p.is_dir()):
        images_dir, labels_dir = class_dir / "images", class_dir / "labels"
        images = list_images(images_dir) if images_dir.exists() else []
        labels = list(labels_dir.glob("*.txt")) if labels_dir.exists() else []
        image_stems = {p.stem for p in images}
        label_stems = {p.stem for p in labels}

        missing.extend((class_dir.name, s) for s in image_stems - label_stems)
        orphan.extend((class_dir.name, s) for s in label_stems - image_stems)

        for image_path in images:
            label_path = labels_dir / f"{image_path.stem}.txt"
            if not label_path.exists():
                continue
            rows = [x.strip() for x in label_path.read_text(encoding="utf-8").splitlines() if x.strip()]
            if not rows:
                empty_count += 1
            for line_no, row in enumerate(rows, 1):
                try:
                    parts = row.split()
                    if len(parts) != 5:
                        raise ValueError("Expected five values")
                    cls = int(float(parts[0]))
                    coords = [float(x) for x in parts[1:]]
                    if cls not in CLASS_MAPPING:
                        raise ValueError(f"Unexpected original class id {cls}")
                    if not all(0 <= x <= 1 for x in coords):
                        raise ValueError("Coordinates outside [0,1]")
                    if coords[2] <= 0 or coords[3] <= 0:
                        raise ValueError("Box width/height must be positive")
                    class_counts[cls] += 1
                except Exception as exc:
                    invalid.append((str(label_path), line_no, row, str(exc)))
            records.append({"source_class": class_dir.name, "image": image_path, "label": label_path})

    return {"records": records, "missing": missing, "orphan": orphan,
            "invalid": invalid, "empty": empty_count, "class_counts": class_counts}

train_audit = audit_split("Train")
val_audit = audit_split("Val")

for name, audit in [("Train", train_audit), ("Val", val_audit)]:
    print(f"\\n{name}: matched={len(audit['records'])}, missing labels={len(audit['missing'])}, "
          f"orphan labels={len(audit['orphan'])}, empty labels={audit['empty']}, "
          f"invalid rows={len(audit['invalid'])}, boxes={dict(audit['class_counts'])}")

print("Invalid examples:", (train_audit["invalid"] + val_audit["invalid"])[:10])
assert not (train_audit["invalid"] or val_audit["invalid"]), "Fix invalid annotations before continuing."

## 6. Verify original class IDs
The earlier project assumed original IDs 0=Glioma, 1=Meningioma, 2=No Tumor, 3=Pituitary. The detector maps IDs 0, 1 and 3 to three tumor classes and ignores ID 2. Confirm this mapping from the dataset documentation before using the trained model.

In [ ]:
print("Original-to-YOLO class mapping:", CLASS_MAPPING)
print("YOLO class names:", dict(enumerate(CLASS_NAMES)))
print("Please verify this mapping against the dataset's official documentation.")

## 7. Visualize annotated examples

In [ ]:
def show_annotated(image_path, label_path):
    image = Image.open(image_path).convert("RGB")
    draw = ImageDraw.Draw(image)
    w, h = image.size
    for row in label_path.read_text(encoding="utf-8").splitlines():
        parts = row.split()
        if len(parts) != 5:
            continue
        old_id = int(float(parts[0]))
        if old_id not in CLASS_MAPPING:
            continue
        new_id = CLASS_MAPPING[old_id]
        xc, yc, bw, bh = map(float, parts[1:])
        x1, y1 = (xc-bw/2)*w, (yc-bh/2)*h
        x2, y2 = (xc+bw/2)*w, (yc+bh/2)*h
        draw.rectangle([x1, y1, x2, y2], outline="red", width=3)
        draw.text((x1, max(0, y1-16)), CLASS_NAMES[new_id], fill="yellow")
    plt.figure(figsize=(7, 7))
    plt.imshow(image)
    plt.axis("off")
    plt.title(image_path.name)
    plt.show()

for record in train_audit["records"][:6]:
    show_annotated(record["image"], record["label"])

## 8. Create train/valid/test datasets
Unmatched images and orphan annotation files are excluded. Empty annotation files are retained for images with no tumor.

In [ ]:
train_pool, test_records = train_test_split(
    train_audit["records"],
    test_size=0.15,
    random_state=SEED,
    stratify=[r["source_class"] for r in train_audit["records"]]
)
split_records = {
    "train": train_pool,
    "valid": val_audit["records"],
    "test": test_records
}

if OUTPUT_DATASET.exists():
    shutil.rmtree(OUTPUT_DATASET)

for split in split_records:
    (OUTPUT_DATASET / "images" / split).mkdir(parents=True, exist_ok=True)
    (OUTPUT_DATASET / "labels" / split).mkdir(parents=True, exist_ok=True)

def convert_label(src, dst):
    converted = []
    for line in src.read_text(encoding="utf-8").splitlines():
        parts = line.split()
        if not parts:
            continue
        old_id = int(float(parts[0]))
        if old_id in CLASS_MAPPING:
            converted.append(" ".join([str(CLASS_MAPPING[old_id])] + parts[1:]))
    dst.write_text("\\n".join(converted) + ("\\n" if converted else ""), encoding="utf-8")

for split, records in split_records.items():
    for record in records:
        src_image, src_label = record["image"], record["label"]
        safe_name = f"{record['source_class'].lower().replace(' ', '_')}__{src_image.name}"
        dst_image = OUTPUT_DATASET / "images" / split / safe_name
        dst_label = OUTPUT_DATASET / "labels" / split / f"{Path(safe_name).stem}.txt"
        shutil.copy2(src_image, dst_image)
        convert_label(src_label, dst_label)

print({split: len(records) for split, records in split_records.items()})

## 9. Audit prepared dataset

In [ ]:
prepared_summary = {}
for split in ["train", "valid", "test"]:
    image_dir = OUTPUT_DATASET / "images" / split
    label_dir = OUTPUT_DATASET / "labels" / split
    images, labels = list_images(image_dir), list(label_dir.glob("*.txt"))
    image_stems, label_stems = {p.stem for p in images}, {p.stem for p in labels}
    empty, boxes, invalid = 0, Counter(), 0

    for label in labels:
        rows = [x.strip() for x in label.read_text(encoding="utf-8").splitlines() if x.strip()]
        if not rows:
            empty += 1
        for row in rows:
            try:
                p = row.split()
                assert len(p) == 5
                cls = int(p[0])
                coords = [float(x) for x in p[1:]]
                assert 0 <= cls < len(CLASS_NAMES)
                assert all(0 <= x <= 1 for x in coords)
                assert coords[2] > 0 and coords[3] > 0
                boxes[CLASS_NAMES[cls]] += 1
            except Exception:
                invalid += 1

    prepared_summary[split] = {
        "images": len(images), "labels": len(labels),
        "missing": len(image_stems-label_stems),
        "orphan": len(label_stems-image_stems),
        "empty_labels": empty, "boxes": dict(boxes), "invalid_rows": invalid
    }
    print(split.upper(), prepared_summary[split])

assert all(x["images"] == x["labels"] and x["missing"] == 0 and
           x["orphan"] == 0 and x["invalid_rows"] == 0
           for x in prepared_summary.values())

## 10. Create data.yaml

In [ ]:
data_config = {
    "path": str(OUTPUT_DATASET.resolve()).replace("\\\\", "/"),
    "train": "images/train",
    "val": "images/valid",
    "test": "images/test",
    "nc": len(CLASS_NAMES),
    "names": CLASS_NAMES
}
DATA_YAML = OUTPUT_DATASET / "data.yaml"
with open(DATA_YAML, "w", encoding="utf-8") as f:
    yaml.safe_dump(data_config, f, sort_keys=False)
print(DATA_YAML.read_text(encoding="utf-8"))

## 11. Configure local MLflow

In [ ]:
MLFLOW_DB = PROJECT_ROOT / "mlflow.db"
mlflow.set_tracking_uri(f"sqlite:///{MLFLOW_DB.as_posix()}")
mlflow.set_experiment("brainscan-v2-detection")
print("Tracking URI:", mlflow.get_tracking_uri())

## 12. Train YOLOv8
This trains a new local run. If you only want to evaluate your existing `models/final/best.pt`, skip this cell and set `BEST_MODEL` in the next section.

In [ ]:
BASE_MODEL = "yolov8n.pt"
train_params = {
    "data": str(DATA_YAML), "epochs": EPOCHS, "imgsz": IMG_SIZE,
    "batch": BATCH_SIZE, "device": DEVICE, "workers": WORKERS,
    "seed": SEED, "patience": 10, "project": str(RUNS_DIR),
    "name": "yolov8n-brain-tumor", "exist_ok": True, "plots": True,
    # Explicit augmentation settings
    "hsv_h": 0.015, "hsv_s": 0.7, "hsv_v": 0.4,
    "degrees": 0.0, "translate": 0.1, "scale": 0.5,
    "shear": 0.0, "perspective": 0.0, "fliplr": 0.5,
    "flipud": 0.0, "mosaic": 1.0, "mixup": 0.0,
    "copy_paste": 0.0, "close_mosaic": 10
}

with mlflow.start_run(run_name="yolov8n-brain-tumor-training") as run:
    mlflow.log_params({
        k: v for k, v in train_params.items()
        if k not in {"project", "exist_ok", "plots"}
    })
    mlflow.log_param("class_names", json.dumps(CLASS_NAMES))
    mlflow.log_param("class_mapping", json.dumps(CLASS_MAPPING))
    detector = YOLO(BASE_MODEL)
    detector.train(**train_params)
    TRAINING_RUN_ID = run.info.run_id

TRAIN_DIR = RUNS_DIR / "yolov8n-brain-tumor"
BEST_MODEL = TRAIN_DIR / "weights" / "best.pt"
assert BEST_MODEL.exists(), f"best.pt missing: {BEST_MODEL}"
print("Best checkpoint:", BEST_MODEL)

## 13. Evaluate validation and test splits
Use validation for development. Use the test split once for final reporting.

In [ ]:
# If evaluating an already-trained model instead of training above, uncomment and edit:
# BEST_MODEL = PROJECT_ROOT / "models" / "final" / "best.pt"

assert BEST_MODEL.exists(), f"Checkpoint not found: {BEST_MODEL}"
best_model = YOLO(str(BEST_MODEL))

def log_metrics(metrics, prefix):
    for key, value in metrics.items():
        safe = re.sub(r"[^A-Za-z0-9_./ :\\-]", "_", key)
        mlflow.log_metric(f"{prefix}_{safe}", float(value))

with mlflow.start_run(run_name="yolov8n-final-evaluation") as eval_run:
    mlflow.log_param("checkpoint", str(BEST_MODEL))
    mlflow.log_param("imgsz", IMG_SIZE)

    val_results = best_model.val(
        data=str(DATA_YAML), split="val", imgsz=IMG_SIZE,
        batch=BATCH_SIZE, device=DEVICE, plots=True,
        project=str(RUNS_DIR / "evaluation"), name="validation", exist_ok=True
    )
    log_metrics(val_results.results_dict, "validation")

    test_results = best_model.val(
        data=str(DATA_YAML), split="test", imgsz=IMG_SIZE,
        batch=BATCH_SIZE, device=DEVICE, plots=True,
        project=str(RUNS_DIR / "evaluation"), name="test", exist_ok=True
    )
    log_metrics(test_results.results_dict, "test")
    mlflow.log_artifact(str(BEST_MODEL), artifact_path="model_weights")
    EVALUATION_RUN_ID = eval_run.info.run_id

print("Validation metrics:")
for k, v in val_results.results_dict.items():
    print(f"{k}: {v:.4f}")
print("\\nTest metrics:")
for k, v in test_results.results_dict.items():
    print(f"{k}: {v:.4f}")
print("MLflow evaluation run:", EVALUATION_RUN_ID)

## 14. Export model, metadata, plots and ZIP

In [ ]:
EXPORT_DIR.mkdir(parents=True, exist_ok=True)
shutil.copy2(BEST_MODEL, EXPORT_DIR / "best.pt")
shutil.copy2(DATA_YAML, EXPORT_DIR / "data.yaml")

metadata = {
    "model_name": "brainscan_yolov8n",
    "architecture": "YOLOv8n",
    "task": "object_detection",
    "class_names": CLASS_NAMES,
    "original_class_mapping": CLASS_MAPPING,
    "image_size": IMG_SIZE,
    "epochs": EPOCHS,
    "batch_size": BATCH_SIZE,
    "seed": SEED,
    "test_metrics": {k: float(v) for k, v in test_results.results_dict.items()}
}
with open(EXPORT_DIR / "metadata.json", "w", encoding="utf-8") as f:
    json.dump(metadata, f, indent=4)

if TRAIN_DIR.exists():
    shutil.copytree(TRAIN_DIR, EXPORT_DIR / "training_results", dirs_exist_ok=True)

EVAL_DIR = RUNS_DIR / "evaluation"
if EVAL_DIR.exists():
    shutil.copytree(EVAL_DIR, EXPORT_DIR / "evaluation_results", dirs_exist_ok=True)

zip_path = shutil.make_archive(str(PROJECT_ROOT / "brainscan_yolo_export"), "zip", EXPORT_DIR)
print("Export folder:", EXPORT_DIR)
print("ZIP:", zip_path)

## 15. Optional inference example on a test image

In [ ]:
test_images = list_images(OUTPUT_DATASET / "images" / "test")
if test_images:
    sample = test_images[0]
    predictions = best_model.predict(
        source=str(sample), imgsz=IMG_SIZE, conf=0.25,
        save=True, project=str(RUNS_DIR / "prediction_examples"),
        name="sample", exist_ok=True
    )
    display(Image.open(sample).convert("RGB"))
    display(Image.fromarray(predictions[0].plot()[:, :, ::-1]))
else:
    print("No test images available.")

## 16. Completion checklist

- [ ] Confirm the original class-ID mapping with dataset documentation.
- [ ] Inspect annotated samples from every tumor class.
- [ ] Check matching image/label counts and invalid annotation rows.
- [ ] Confirm empty labels are genuine no-tumor images.
- [ ] Train and preserve the actual hyperparameters/augmentation settings.
- [ ] Evaluate on validation and test; report test metrics separately.
- [ ] Preserve `best.pt`, `data.yaml`, `metadata.json`, plots and MLflow runs.
- [ ] Check patient-level overlap if patient identifiers are available.
- [ ] Copy the final `best.pt` into the main project's `models/final/`.

**Reporting:** YOLO detection metrics and InceptionV3 classification metrics measure different tasks. Present them separately. This is an educational/research prototype, not a clinical diagnostic system.